In [16]:
!pip install streamlit pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 7.8 MB/s eta 0:00:00


In [ ]:
!pip install -U layoutparser

In [2]:
!apt install poppler-utils
!pip install pdf2image

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.8).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [3]:
!pip install 'git+https://github.com/facebookresearch/detectron2.git@v0.4#egg=detectron2'

  Cloning https://github.com/facebookresearch/detectron2.git (to revision v0.4) to /tmp/pip-install-j2292lgn/detectron2_b6ec297258e440cbb67ab681684355f2
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-install-j2292lgn/detectron2_b6ec297258e440cbb67ab681684355f2
  Running command git checkout -q 4aca4bdaa9ad48b8e91d7520e0d0815bb8ca0fb1
  Resolved https://github.com/facebookresearch/detectron2.git to commit 4aca4bdaa9ad48b8e91d7520e0d0815bb8ca0fb1
  Preparing metadata (setup.py) ... done


In [4]:
!pip install layoutparser[ocr]

In [5]:
!git clone https://github.com/Layout-Parser/layout-parser.git

fatal: destination path 'layout-parser' already exists and is not an empty directory.


In [6]:
%cd layout-parser/

/content/layout-parser


In [8]:
!pip install pillow==9.5.0

In [ ]:
%%writefile pdf_new.py
import streamlit as st
import tempfile
import cv2
import os
import numpy as np
from pdf2image import convert_from_bytes
import layoutparser as lp
import zipfile

# Function to process the uploaded PDF file
def process_pdf(uploaded_file):
    # Create a temporary working directory
    with tempfile.TemporaryDirectory() as tmpdir:
        # Read the uploaded PDF file as bytes
        pdf_bytes = uploaded_file.read()
        # Convert PDF pages to images
        pages = convert_from_bytes(pdf_bytes)

        # Create output folders for different content types
        output_folders = ["Text", "Figure", "Table"]
        for folder in output_folders:
            os.makedirs(os.path.join(tmpdir, folder), exist_ok=True)

        # Load the LayoutParser model for detecting layout elements
        model = lp.Detectron2LayoutModel(
            'lp://PubLayNet/mask_rcnn_X_101_32x8d_FPN_3x/config',
            extra_config=["MODEL.ROI_HEADS.SCORE_THRESH_TEST", 0.5],
            label_map={0: "Text", 1: "Title", 2: "List", 3: "Table", 4: "Figure"}
        )

        # Initialize counters for each content type
        count = {label: 0 for label in output_folders}

        # Process each page of the PDF
        for page_num, page_image in enumerate(pages):
            # Convert the page image to OpenCV format (BGR)
            image = cv2.cvtColor(np.array(page_image), cv2.COLOR_RGB2BGR)
            # Detect layout elements on the page
            layout = model.detect(image)

            # Process each detected layout block
            for block in layout:
                label = block.type
                # Skip blocks that are not in the output folders
                if label not in output_folders:
                    continue

                # Extract the coordinates of the block
                x1, y1, x2, y2 = map(int, block.coordinates)
                # Crop the block from the image
                cropped_image = image[y1:y2, x1:x2]
                # Save the cropped image to the corresponding folder
                save_path = os.path.join(tmpdir, label, f"{label}_{count[label]}_page{page_num+1}.png")
                cv2.imwrite(save_path, cropped_image)
                # Increment the counter for the label
                count[label] += 1

        # Create a ZIP file containing all the output folders
        # Define the path for the output ZIP file
        zip_path = os.path.join(tmpdir, "output.zip")
        
        # Create a ZIP file to store all the extracted content
        with zipfile.ZipFile(zip_path, 'w') as zipf:
            # Iterate through each output folder (Text, Figure, Table)
            for folder in output_folders:
            # Get the full path of the current folder
            folder_path = os.path.join(tmpdir, folder)
            # Walk through the folder to retrieve all files
            for root, _, files in os.walk(folder_path):
                for file in files:
                # Get the full path of the current file
                file_path = os.path.join(root, file)
                # Calculate the relative path of the file to maintain folder structure in the ZIP
                arcname = os.path.relpath(file_path, tmpdir)
                supervised learning\neural_networks\YOLO\Document_Analysis_App.ipynb
                # Add the file to the ZIP archive
                zipf.write(file_path, arcname)
                zipf\supervised learning\neural_networks\YOLO\Document_Analysis_App.ipynb

        # Provide the ZIP file as a download button in the Streamlit app
        with open(zip_path, "rb") as f:
            st.download_button(
                label="Download Extracted Files (ZIP)",
                data=f,
                file_name='output.zip',
                mime='application/zip'
            )

# Streamlit UI
st.title("PDF to Text, Tables, and Figures Extractor")
# File uploader for the user to upload a PDF
uploaded_file = st.file_uploader("Upload a PDF", type=["pdf"])

# Process the uploaded file if it exists
if uploaded_file is not None:
    process_pdf(uploaded_file)
else:
    # Display a message if no file is uploaded
    st.write("Please upload a PDF to begin.")


Overwriting pdf_new.py


In [33]:
from pyngrok import ngrok

# Add your ngrok authtoken properly using pyngrok API (no !)
ngrok.set_auth_token("2wzlDuYRiyurvfmfSUCsSzadd0X_5dA1QzwfZCbxA4htoa5j5")

# Set up a tunnel to the Streamlit app on port 8501
public_url = ngrok.connect(8501)
print(f"Streamlit app is live at: {public_url}")

Streamlit app is live at: NgrokTunnel: "https://aebb-104-196-162-30.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
# Run the Streamlit app in the background and redirect logs to a file
!streamlit run pdf_new.py &>/content/logs.txt &
